# Alternating discovery and refinement cycles (50D)

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Section 5.7, Appendix N.1, Table 20.

Repeats Stage II (rediscovery on the current surrogate) and Stage III (refinement with frozen generators) for several cycles and reports drift/diffusion MSE and the within-cycle determining-loss reduction Δ_k.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Config, imports, and E3 controls

In [ ]:
# ============================================================
# Cell 1 — Config, imports, and E3 controls
# ============================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import csv
import io
import math
import time
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


CONFIG = {
    # Reproducibility
    "SEED": 42,

    # ----------------------------
    # 50D synthetic SDE generation
    # ----------------------------
    "STATE_DIM": 50,

    # Choose one:
    #   "ring_double_well_50d"
    #   "low_rank_ou_50d"
    #   "block_lorenz_50d"
    #   "tanh_chain_50d"
    "TRUE_SDE_NAME": "tanh_chain_50d",

    "T_FINAL": 2.0,
    "DT": 0.01,
    "N_TRAJECTORIES": 64,
    "X0_MEAN": 0.0,
    "X0_STD": 1.0,
    "TRAIN_FRACTION": 0.8,

    # ----------------------------
    # Neural SDE surrogate
    # ----------------------------
    "MLP_HIDDEN_DIM": 192,
    "MLP_DEPTH": 3,
    "INPUT_T_SCALE": 2.0,
    "INPUT_X_SCALE": 3.0,
    "SIGMA_EPS": 1e-4,

    # ----------------------------
    # Stage I: vanilla surrogate
    # ----------------------------
    "BASELINE_EPOCHS": 2000,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-3,
    "PRINT_EVERY": 250,

    # ----------------------------
    # Stage II: symmetry discovery
    # ----------------------------
    # E3 should first be run with a small, practical generator bank.
    # The d+2 result is an upper bound, not a requirement to use 52 generators.
    # ----------------------------
    # Stage II: symmetry discovery
    # ----------------------------
    # Use the submitted 50D generator count so that cycle 1 exactly reproduces
    # the submitted single-pass protocol. The d+2=52 result is an upper bound on
    # projectable symmetry-algebra dimension, not a requirement to use 52.
    "N_GENERATORS": 50,
    "SYM_DISCOVERY_EPOCHS": 4000,
    "SYM_DISCOVERY_LR": 5e-3,
    "SYM_DISCOVERY_POINTS": 256,
    "SYM_DISCOVERY_T_BOX": (0.0, 2.0),
    "SYM_DISCOVERY_X_BOX": (-3.0, 3.0),
    "SYM_INIT_SCALE": 0.03,
    "SYM_JITTER_STD": 0.05,

    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.5,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.10,
    "CLOSURE_RCOND": 1e-5,

    # ----------------------------
    # 50D basis library
    # ----------------------------
    "BASIS_MODE": "expanded_gaeta_lowrank",
    "BASIS_LINEAR_MU_LIST": [-1.0, -0.5, 0.5, 1.0],
    "BASIS_EXP_CLIP": 6.0,
    "BASIS_LOW_RANK_K": 6,
    "BASIS_ADD_EXPONENTIAL": True,
    "BASIS_ADD_LOW_RANK_ROTATIONS": True,
    "BASIS_ADD_LOW_RANK_SHEARS": True,

    # ----------------------------
    # Stage III: frozen-W refinement
    # ----------------------------
    "STAGE_F_EPOCHS": 800,
    "STAGE_F_BATCH_SIZE": 512,
    "STAGE_F_LR": 3e-4,
    "STAGE_F_LAMBDA_SYM_DET": 105000.0,
    "STAGE_F_LAMBDA_PROJECTABLE": 1e-3,
    "STAGE_F_N_EXTRA_SYM_POINTS": 128,
    "STAGE_F_N_DATA_SYM_POINTS": 128,
    "STAGE_F_EXTRA_T_BOX": (0.0, 2.0),
    "STAGE_F_EXTRA_X_BOX": (-3.0, 3.0),

    # ----------------------------
    # E3 alternating-loop protocol
    # ----------------------------
    "E3_N_CYCLES": 6,
    "E3_DISCOVERY_INIT": "fresh",

    # These base seeds make the sampled optimization schedule identical
    # across cycles. Only W initialization changes with cycle index.
    "E3_DISCOVERY_INIT_SEED": 93000,
    "E3_DISCOVERY_SAMPLE_SEED": 93001,
    "E3_REFINEMENT_BATCH_SEED": 93002,
    "E3_REFINEMENT_EXTRA_SEED": 93003,

    # Fixed held-out evaluation clouds, reused at every cycle.
    "E3_FIXED_EVAL_SETS": True,
    "E3_PRIMARY_EVAL_DISTRIBUTION": "heldout_trajectory_plus_jitter",
    "E3_PRIMARY_EVAL_POINTS": 2000,
    "E3_PRIMARY_EVAL_JITTER_STD": 0.05,
    "E3_BOX_EVAL_POINTS": 2000,
    "E3_RESIDUAL_EVAL_POINTS": 128,
    "E3_SUBSPACE_EVAL_POINTS": 256,
    "E3_EVAL_SEED": 94001,
    "E3_RESIDUAL_SEED": 94002,
    "E3_SUBSPACE_SEED": 94003,

    # Numerical rank for function-space generator subspaces.
    "E3_SUBSPACE_RCOND": 1e-6,

    # Explicit interpretation tolerances used by the final verdict.
    "E3_PLATEAU_REL_TOL": 0.01,
    "E3_DEGRADE_REL_TOL": 0.03,
    "E3_STABLE_ANGLE_MEAN_DEG": 10.0,
    "E3_STABLE_ANGLE_MAX_DEG": 20.0,

    # Optional diagnostics/plots
    "N_EVAL_POINTS": 2000,
    "N_PLOT_GRID": 160,
    "PLOT_COMPONENTS": [0, 1, 2, 9, 24, 49],
}

D = CONFIG["STATE_DIM"]
Q = D + 1
key = jax.random.PRNGKey(CONFIG["SEED"])

if CONFIG["STATE_DIM"] == 50 and CONFIG["N_GENERATORS"] != 50:
    raise ValueError(
        "The reportable 50D E3 run must use N_GENERATORS=50 so that "
        "cycle 1 matches the submitted single-pass configuration. "
        "Use a separate explicitly labeled pilot notebook for smaller m."
    )

## Cell 2 — Editable 50D SDE registry and synthetic data

In [ ]:
# ============================================================
# Cell 2 — Editable 50D SDE registry and synthetic data
# ============================================================

# ------------------------------------------------------------------
# How to use:
#
# Change CONFIG["TRUE_SDE_NAME"] in Cell 1.
#
# Available examples:
#   "ring_double_well_50d"
#   "low_rank_ou_50d"
#   "block_lorenz_50d"
#   "tanh_chain_50d"
#
# Each SDE entry defines:
#   drift_fn(t, x) -> shape (..., 50)
#   sigma_fn(t, x) -> shape (..., 50), positive diagonal diffusion
# ------------------------------------------------------------------


def _idx_float():
    return jnp.arange(D, dtype=jnp.float32)


def _phase_vector():
    return 2.0 * jnp.pi * _idx_float() / float(D)


def _mode_sin(k):
    return jnp.sin(2.0 * jnp.pi * k * _idx_float() / float(D))


def _mode_cos(k):
    return jnp.cos(2.0 * jnp.pi * k * _idx_float() / float(D))


def _normalize_mode(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


# ============================================================
# 50D SDE example 1: ring-coupled double-well
# ============================================================

def drift_ring_double_well_50d(t, x):
    """
    50D nonlinear ring-coupled double-well SDE.

    Drift:
        f_i = x_i - x_i^3
              + c (x_{i-1} + x_{i+1} - 2 x_i)
              + weak time forcing_i(t)

    Nontrivial features:
        - nonlinear bistability in every coordinate
        - local spatial coupling on a ring
        - weak heterogeneous forcing
    """
    c = 0.25
    lap = jnp.roll(x, 1, axis=-1) + jnp.roll(x, -1, axis=-1) - 2.0 * x

    phase = _phase_vector()
    forcing = 0.08 * jnp.sin(2.0 * jnp.pi * t + phase)

    return x - x**3 + c * lap + forcing


def sigma_ring_double_well_50d(t, x):
    """
    Positive state-dependent diagonal diffusion.
    """
    phase = _phase_vector()
    base = 0.18 + 0.02 * jnp.sin(phase) ** 2
    return base + 0.04 * jax.nn.sigmoid(x)


# ============================================================
# 50D SDE example 2: low-rank coupled OU/spiral system
# ============================================================

def drift_low_rank_ou_50d(t, x):
    """
    Stable 50D linear/nonlinear low-rank OU-like SDE.

    Drift:
        stable coordinate damping
        + skew ring coupling
        + low-rank global coupling
        + weak periodic forcing
        + small tanh nonlinearity

    This is useful because the true drift has global low-rank structure
    but is still learned by a generic neural SDE.
    """
    idx = _idx_float()

    # Coordinate-dependent damping
    lam = 0.45 + 0.30 * (idx / float(D - 1))

    # Ring skew/transport-like coupling
    skew = 0.18 * (jnp.roll(x, -1, axis=-1) - jnp.roll(x, 1, axis=-1))

    v1 = _normalize_mode(_mode_sin(1))
    v2 = _normalize_mode(_mode_cos(2))
    v3 = _normalize_mode(_mode_sin(3))

    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)
    p3 = jnp.sum(x * v3, axis=-1, keepdims=True)

    low_rank = (
        0.45 * p1 * v2
        - 0.35 * p2 * v1
        + 0.25 * jnp.tanh(p3) * v3
    )

    forcing = 0.05 * (
        jnp.sin(1.2 * t) * v1
        + jnp.cos(0.8 * t) * v2
    )

    return -lam * x + skew + low_rank + forcing + 0.05 * jnp.tanh(x)


def sigma_low_rank_ou_50d(t, x):
    v1 = _normalize_mode(_mode_sin(1))
    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    base = 0.20 + 0.03 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jax.nn.sigmoid(p1) + 0.02 * jnp.tanh(x) ** 2


# ============================================================
# 50D SDE example 3: block Lorenz-style system
# ============================================================

def drift_block_lorenz_50d(t, x):
    """
    50D block-tamed Lorenz-style SDE.

    First 48 coordinates form 16 weakly coupled Lorenz-like triples.
    Last 2 coordinates are stable auxiliary variables coupled to block means.

    The Lorenz blocks are tamed to avoid numerical explosions.
    """
    x_blocks = x[..., :48].reshape(x.shape[:-1] + (16, 3))
    y = x[..., 48:50]

    x1 = x_blocks[..., 0]
    x2 = x_blocks[..., 1]
    x3 = x_blocks[..., 2]

    sigma_l = 4.0
    rho = 8.0
    beta = 8.0 / 3.0

    g1 = sigma_l * (x2 - x1)
    g2 = x1 * (rho - x3) - x2
    g3 = x1 * x2 - beta * x3

    raw = jnp.stack([g1, g2, g3], axis=-1)

    norm2 = jnp.sum(x_blocks**2, axis=-1, keepdims=True)
    tame = 1.0 + 0.05 * norm2

    block_f = raw / tame

    # Weak coupling between neighboring blocks
    block_lap = (
        jnp.roll(x_blocks, 1, axis=-2)
        + jnp.roll(x_blocks, -1, axis=-2)
        - 2.0 * x_blocks
    )
    block_f = block_f + 0.05 * block_lap

    # Auxiliary coordinates
    block_mean = jnp.mean(x_blocks, axis=(-2, -1), keepdims=False)
    f_y1 = -0.60 * y[..., 0] + 0.10 * jnp.tanh(block_mean)
    f_y2 = -0.75 * y[..., 1] + 0.10 * jnp.sin(t)

    f_y = jnp.stack([f_y1, f_y2], axis=-1)

    return jnp.concatenate([block_f.reshape(x.shape[:-1] + (48,)), f_y], axis=-1)


def sigma_block_lorenz_50d(t, x):
    base = 0.12 + 0.02 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jnp.tanh(x) ** 2


# ============================================================
# 50D SDE example 4: nonlinear tanh reaction chain
# ============================================================

def drift_tanh_chain_50d(t, x):
    """
    50D smooth nonlinear reaction-chain SDE.

    Drift:
        damping
        + forward/backward tanh couplings
        + weak global low-rank feedback
        + heterogeneous forcing

    This is bounded and stable, but still nonlinear and coupled.
    """
    phase = _phase_vector()

    f_local = (
        -0.55 * x
        + 0.35 * jnp.tanh(jnp.roll(x, -1, axis=-1))
        - 0.25 * jnp.tanh(jnp.roll(x, 1, axis=-1))
        + 0.10 * jnp.tanh(jnp.roll(x, -2, axis=-1))
    )

    v1 = _normalize_mode(_mode_cos(1))
    v2 = _normalize_mode(_mode_sin(2))

    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)

    global_feedback = 0.20 * jnp.tanh(p1) * v2 - 0.15 * jnp.tanh(p2) * v1

    forcing = 0.04 * jnp.sin(1.5 * t + phase)

    return f_local + global_feedback + forcing


def sigma_tanh_chain_50d(t, x):
    return 0.22 + 0.04 * jnp.sin(x) ** 2


# ============================================================
# Registry
# ============================================================

SDE_EXAMPLES = {
    "ring_double_well_50d": {
        "description": "50D nonlinear ring-coupled double-well SDE with state-dependent diffusion.",
        "drift_fn": drift_ring_double_well_50d,
        "sigma_fn": sigma_ring_double_well_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.75,
        "input_x_scale": 2.5,
        "x_box": (-2.5, 2.5),
    },

    "low_rank_ou_50d": {
        "description": "50D stable low-rank OU/spiral SDE with global modes and state-dependent diffusion.",
        "drift_fn": drift_low_rank_ou_50d,
        "sigma_fn": sigma_low_rank_ou_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },

    "block_lorenz_50d": {
        "description": "50D tamed Lorenz-block SDE: 16 coupled triples plus 2 stable auxiliary coordinates.",
        "drift_fn": drift_block_lorenz_50d,
        "sigma_fn": sigma_block_lorenz_50d,
        "t_final": 1.0,
        "dt": 0.005,
        "x0_mean": 0.0,
        "x0_std": 0.8,
        "input_x_scale": 5.0,
        "x_box": (-5.0, 8.0),
    },

    "tanh_chain_50d": {
        "description": "50D damped nonlinear tanh reaction chain with low-rank feedback.",
        "drift_fn": drift_tanh_chain_50d,
        "sigma_fn": sigma_tanh_chain_50d,
        "t_final": 3.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },
}


def activate_sde_from_config():
    """
    Select active SDE from CONFIG["TRUE_SDE_NAME"] and update
    dependent CONFIG fields automatically.
    """
    name = CONFIG["TRUE_SDE_NAME"]

    if name not in SDE_EXAMPLES:
        raise ValueError(
            f"Unknown TRUE_SDE_NAME={name}. "
            f"Available choices: {list(SDE_EXAMPLES.keys())}"
        )

    spec = SDE_EXAMPLES[name]

    CONFIG["T_FINAL"] = float(spec["t_final"])
    CONFIG["DT"] = float(spec["dt"])
    CONFIG["X0_MEAN"] = spec["x0_mean"]
    CONFIG["X0_STD"] = spec["x0_std"]
    CONFIG["INPUT_T_SCALE"] = float(spec["t_final"])
    CONFIG["INPUT_X_SCALE"] = float(spec["input_x_scale"])

    x_box = tuple(float(v) for v in spec["x_box"])

    CONFIG["SYM_DISCOVERY_T_BOX"] = (0.0, float(spec["t_final"]))
    CONFIG["STAGE_F_EXTRA_T_BOX"] = (0.0, float(spec["t_final"]))

    CONFIG["SYM_DISCOVERY_X_BOX"] = x_box
    CONFIG["STAGE_F_EXTRA_X_BOX"] = x_box

    return spec


ACTIVE_SDE = activate_sde_from_config()


def _as_50d_vector(value, name):
    """
    Convert scalar or length-D array/tuple into shape (D,).
    """
    arr = jnp.asarray(value, dtype=jnp.float32)

    if arr.ndim == 0:
        return jnp.full((D,), arr, dtype=jnp.float32)

    if arr.shape != (D,):
        raise ValueError(f"{name} must be scalar or shape ({D},), got {arr.shape}")

    return arr


def true_drift(t, x):
    """
    Dispatch to active 50D drift.

    x shape:
        (..., 50)

    return shape:
        (..., 50)
    """
    return ACTIVE_SDE["drift_fn"](t, x)


def true_sigma(t, x):
    """
    Dispatch to active 50D diagonal diffusion.

    x shape:
        (..., 50)

    return shape:
        (..., 50)
    """
    sigma = ACTIVE_SDE["sigma_fn"](t, x)
    return jnp.maximum(sigma, 1e-6)


def print_active_sde_report():
    print("\n" + "=" * 72)
    print("ACTIVE 50D SYNTHETIC SDE")
    print("=" * 72)
    print(f"TRUE_SDE_NAME: {CONFIG['TRUE_SDE_NAME']}")
    print(f"Description  : {ACTIVE_SDE['description']}")
    print(f"STATE_DIM    : {D}")
    print(f"T_FINAL      : {CONFIG['T_FINAL']}")
    print(f"DT           : {CONFIG['DT']}")
    print(f"X0_MEAN      : {CONFIG['X0_MEAN']}")
    print(f"X0_STD       : {CONFIG['X0_STD']}")
    print(f"X_BOX        : {CONFIG['STAGE_F_EXTRA_X_BOX']}")
    print(f"INPUT_T_SCALE: {CONFIG['INPUT_T_SCALE']}")
    print(f"INPUT_X_SCALE: {CONFIG['INPUT_X_SCALE']}")
    print("=" * 72)


print_active_sde_report()


@jax.jit
def simulate_sde_paths(key):
    """
    Simulate all trajectories using Euler-Maruyama.

    Returns
    -------
    times : shape (n_steps + 1,)
    paths : shape (n_paths, n_steps + 1, 50)
    """
    T = CONFIG["T_FINAL"]
    dt = CONFIG["DT"]
    n_steps = int(round(T / dt))
    n_paths = CONFIG["N_TRAJECTORIES"]

    key_x0, key_noise = jax.random.split(key)

    x0_mean = _as_50d_vector(CONFIG["X0_MEAN"], "X0_MEAN")
    x0_std = _as_50d_vector(CONFIG["X0_STD"], "X0_STD")

    x0 = x0_mean[None, :] + x0_std[None, :] * jax.random.normal(
        key_x0,
        (n_paths, D),
    )

    noise = jax.random.normal(key_noise, (n_steps, n_paths, D))

    times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, zeta = inp
        f = true_drift(t, x)
        s = true_sigma(t, x)
        x_next = x + f * dt + s * sqrt_dt * zeta
        return x_next, x_next

    _, xs_tail = jax.lax.scan(step, x0, (times[:-1], noise))

    paths = jnp.concatenate([x0[None, :, :], xs_tail], axis=0)
    paths = jnp.transpose(paths, (1, 0, 2))

    return times, paths


def make_increment_dataset(times, paths):
    """
    Convert simulated trajectories into one-step increment data.

    Returns:
        t0 : shape (n_samples,)
        x0 : shape (n_samples, 50)
        dt : shape (n_samples,)
        dx : shape (n_samples, 50)
    """
    n_paths, n_times, dim = paths.shape
    n_steps = n_times - 1
    dt = CONFIG["DT"]

    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1, :].reshape(-1, dim)
    dx = (paths[:, 1:, :] - paths[:, :-1, :]).reshape(-1, dim)
    dt_arr = jnp.full((x0.shape[0],), dt, dtype=jnp.float32)

    return t0, x0, dt_arr, dx


def train_test_split_paths(key, paths, train_fraction):
    """
    Split complete simulated trajectories into disjoint training and test sets.

    Every one-step transition from a trajectory remains entirely within either
    the training set or the test set. This prevents transitions from the same
    stochastic path from appearing in both sets.

    Parameters
    ----------
    key : JAX PRNG key
        Key used to shuffle trajectory indices reproducibly.

    paths : array, shape (n_paths, n_times, D)
        Complete simulated trajectories.

    train_fraction : float
        Fraction of complete trajectories assigned to training.

    Returns
    -------
    train_paths : array
        Complete trajectories assigned to training.

    test_paths : array
        Complete trajectories assigned to testing.

    train_path_indices : array
        Original trajectory indices assigned to training.

    test_path_indices : array
        Original trajectory indices assigned to testing.
    """
    n_paths = int(paths.shape[0])

    if n_paths < 2:
        raise ValueError(
            "Trajectory-level splitting requires at least two trajectories."
        )

    n_train_paths = int(math.floor(float(train_fraction) * n_paths))

    # Guarantee that both partitions contain at least one complete trajectory.
    n_train_paths = max(1, min(n_train_paths, n_paths - 1))

    permutation = jax.random.permutation(key, n_paths)

    train_path_indices = permutation[:n_train_paths]
    test_path_indices = permutation[n_train_paths:]

    train_paths = paths[train_path_indices]
    test_paths = paths[test_path_indices]

    return (
        train_paths,
        test_paths,
        train_path_indices,
        test_path_indices,
    )


def sample_batch(key, dataset, batch_size):
    n = dataset[0].shape[0]
    idx = jax.random.randint(key, (batch_size,), 0, n)
    return tuple(arr[idx] for arr in dataset)


# ----------------------------
# Generate data and split by complete trajectory
# ----------------------------
key, simulation_key = jax.random.split(key)
times, paths = simulate_sde_paths(simulation_key)

key, split_key = jax.random.split(key)
(
    train_paths,
    test_paths,
    train_path_indices,
    test_path_indices,
) = train_test_split_paths(
    split_key,
    paths,
    CONFIG["TRAIN_FRACTION"],
)

# Flatten transitions only after the complete trajectories have been split.
train_data = make_increment_dataset(times, train_paths)
test_data = make_increment_dataset(times, test_paths)

# Retain the full pooled dataset only as an optional diagnostic object.
# It is not used for training or held-out evaluation.
dataset = make_increment_dataset(times, paths)

print("\n" + "=" * 72)
print("TRAJECTORY-LEVEL TRAIN/TEST SPLIT")
print("=" * 72)
print("times shape              :", times.shape)
print("all paths shape          :", paths.shape)
print("training paths shape     :", train_paths.shape)
print("test paths shape         :", test_paths.shape)
print("number of training paths :", int(train_paths.shape[0]))
print("number of test paths     :", int(test_paths.shape[0]))
print("training transitions     :", int(train_data[0].shape[0]))
print("test transitions         :", int(test_data[0].shape[0]))
print(
    "path sets disjoint       :",
    bool(
        set(np.asarray(train_path_indices).tolist()).isdisjoint(
            set(np.asarray(test_path_indices).tolist())
        )
    ),
)
print("training path indices    :", np.asarray(train_path_indices))
print("test path indices        :", np.asarray(test_path_indices))
print("=" * 72)


# ----------------------------
# Quick trajectory plots for selected components only
# ----------------------------
for comp in CONFIG["PLOT_COMPONENTS"]:
    plt.figure(figsize=(8, 4))
    for i in range(min(12, CONFIG["N_TRAJECTORIES"])):
        plt.plot(np.array(times), np.array(paths[i, :, comp]), alpha=0.6)
    plt.xlabel("t")
    plt.ylabel(f"X_{comp+1}(t)")
    plt.title(
        f"{CONFIG['TRUE_SDE_NAME']} trajectories: component {comp+1}"
    )
    plt.grid(True)
    plt.show()


# ----------------------------
# Sanity check drift/diffusion values
# ----------------------------
key, subkey = jax.random.split(key)
x_probe = jax.random.normal(subkey, (8, D))
t_probe = 0.5 * CONFIG["T_FINAL"]

print("\nDrift probe shape:", true_drift(t_probe, x_probe).shape)
print("Sigma probe shape:", true_sigma(t_probe, x_probe).shape)
print("Sigma probe min  :", float(jnp.min(true_sigma(t_probe, x_probe))))

## Cell 3 — Neural SDE, Adam, and Stage-I checkpoint

In [ ]:
# ============================================================
# Cell 3 — Neural SDE, Adam, and Stage-I checkpoint
# ============================================================

def glorot_uniform(key, in_dim, out_dim):
    lim = jnp.sqrt(6.0 / (in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-lim, maxval=lim)


class MLP:
    """Pure-JAX MLP used for the 50D drift and diagonal diffusion."""

    def __init__(self, in_dim, hidden_dim, depth, out_dim):
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.depth = depth
        self.out_dim = out_dim

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)
        params = []
        for k, in_d, out_d in zip(keys, sizes[:-1], sizes[1:]):
            params.append({
                "W": glorot_uniform(k, in_d, out_d),
                "b": jnp.zeros((out_d,), dtype=jnp.float32),
            })
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        return h @ params[-1]["W"] + params[-1]["b"]


surrogate_model = MLP(
    in_dim=Q,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2 * D,
)


def normalize_input(t, x):
    tn = jnp.asarray([t / CONFIG["INPUT_T_SCALE"]], dtype=jnp.float32)
    xn = x / CONFIG["INPUT_X_SCALE"]
    return jnp.concatenate([tn, xn], axis=0)


def surrogate_f_sigma(params, t, x):
    raw = surrogate_model.apply(params, normalize_input(t, x))
    f = raw[:D]
    sigma = jax.nn.softplus(raw[D:]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def f_vec_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1:])[0]


def sigma_vec_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1:])[1]


def data_nll_loss(params, batch):
    """Euler-Maruyama diagonal-Gaussian transition NLL."""
    t0, x0, dt, dx = batch
    f, sigma = jax.vmap(
        lambda tt, xx: surrogate_f_sigma(params, tt, xx)
    )(t0, x0)

    mean = f * dt[:, None]
    var = sigma**2 * dt[:, None] + 1e-8
    components = 0.5 * (
        (dx - mean) ** 2 / var + jnp.log(2.0 * jnp.pi * var)
    )
    return jnp.mean(jnp.sum(components, axis=-1))


def adam_init(params):
    return {
        "m": jax.tree_util.tree_map(jnp.zeros_like, params),
        "v": jax.tree_util.tree_map(jnp.zeros_like, params),
        "t": jnp.array(0, dtype=jnp.int32),
    }


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1
    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"], grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state["v"], grads,
    )
    m_hat = jax.tree_util.tree_map(
        lambda m: m / (1.0 - beta1**t_new), m_new
    )
    v_hat = jax.tree_util.tree_map(
        lambda v: v / (1.0 - beta2**t_new), v_new
    )
    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * m / (jnp.sqrt(v) + eps),
        params, m_hat, v_hat,
    )
    return params_new, {"m": m_new, "v": v_new, "t": t_new}


def tree_copy(tree):
    """Materialize an independent immutable-style checkpoint."""
    return jax.tree_util.tree_map(lambda x: jnp.array(x), tree)


def tree_block_until_ready(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    if leaves:
        leaves[0].block_until_ready()
    return tree


@jax.jit
def baseline_train_step(params, opt_state, batch):
    loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
    params, opt_state = adam_update(
        params, grads, opt_state, CONFIG["BASELINE_LR"]
    )
    return params, opt_state, loss


# ----------------------------
# Train Stage I once
# ----------------------------
key, init_key = jax.random.split(key)
sde_params = surrogate_model.init(init_key)
sde_opt_state = adam_init(sde_params)

baseline_losses = []
_stage1_t0 = time.perf_counter()

for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
    key, batch_key = jax.random.split(key)
    batch = sample_batch(batch_key, train_data, CONFIG["BASELINE_BATCH_SIZE"])
    sde_params, sde_opt_state, loss = baseline_train_step(
        sde_params, sde_opt_state, batch
    )
    baseline_losses.append(float(loss))

    if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
        train_loss = float(data_nll_loss(sde_params, train_data))
        test_loss = float(data_nll_loss(sde_params, test_data))
        print(
            f"[Stage I] epoch {epoch:5d} | "
            f"batch NLL {float(loss): .6f} | "
            f"train NLL {train_loss: .6f} | "
            f"test NLL {test_loss: .6f}"
        )

_tree_ready = tree_block_until_ready(sde_params)
stage1_seconds = time.perf_counter() - _stage1_t0

theta_0 = tree_copy(sde_params)
sde_params_baseline = theta_0

print(f"Stage-I training time: {stage1_seconds:.2f} sec")
print("Saved immutable Stage-I checkpoint theta_0.")


## Cell 4 — Editable 50D Gaeta-style expanded symmetry-basis library

In [ ]:
# ============================================================
# Cell 4 — Editable 50D Gaeta-style expanded symmetry-basis library
# ============================================================

# ------------------------------------------------------------------
# Each basis returns a 51-vector:
#     [tau, xi1, ..., xi50]
#
# Minimal 104-basis:
#   3 time-only generators:
#       ∂t, t∂t, t^2∂t
#   50 coordinate translations:
#       ∂x_i
#   50 coordinate scalings:
#       x_i ∂x_i
#   1 global Brownian/Gaeta scaling:
#       2(t/T)∂t + x·∇x
#
# Expanded Gaeta/low-rank additions:
#   1. Exponential translations:
#       e^{mu t/T} ∂x_i
#
#   2. Exponential projective linear-drift generators:
#       e^{2mu t/T}(∂t + mu x·∇x)
#
#   3. Low-rank rotations:
#       (v_b·x)v_a·∇x - (v_a·x)v_b·∇x
#
#   4. Low-rank shears:
#       (v_b·x)v_a·∇x
#
# The added basis elements are all at most linear in x, so xi_xx = 0.
# Therefore the Hessian-free determining-equation implementation remains valid.
# ------------------------------------------------------------------

BASIS_LIBRARY = []


def add_basis(name, source, fn):
    BASIS_LIBRARY.append({"name": name, "source": source, "fn": fn})


def time_basis(t):
    """
    Normalized time variable.

    We use t/T rather than raw t inside exponentials and polynomial
    time basis functions for numerical stability.
    """
    return t / CONFIG["INPUT_T_SCALE"]


def safe_exp(z):
    c = CONFIG["BASIS_EXP_CLIP"]
    return jnp.exp(jnp.clip(z, -c, c))


def zero_xi():
    return jnp.zeros((D,), dtype=jnp.float32)


def e_i(i):
    v = jnp.zeros((D,), dtype=jnp.float32)
    return v.at[i].set(1.0)


def unit(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


def make_low_rank_directions(K):
    """
    Generic low-rank directions in coordinate-index space.

    These are not fitted from the data and do not use the true SDE.
    They are fixed Fourier-like directions:
        constant, sin(1), cos(1), sin(2), cos(2), ...
    """
    idx = jnp.arange(D, dtype=jnp.float32)
    dirs = []

    # Global mean direction.
    dirs.append(unit(jnp.ones((D,), dtype=jnp.float32)))

    freq = 1
    while len(dirs) < K:
        phase = 2.0 * jnp.pi * freq * (idx + 0.5) / float(D)

        dirs.append(unit(jnp.sin(phase)))

        if len(dirs) < K:
            dirs.append(unit(jnp.cos(phase)))

        freq += 1

    return tuple(dirs[:K])


# ============================================================
# Group A: time-only projectable basis
# ============================================================

add_basis(
    "time_translation__dt",
    "Generic projectable time translation",
    lambda t, x: jnp.concatenate(
        [jnp.array([1.0], dtype=jnp.float32), zero_xi()]
    ),
)

add_basis(
    "time_scaling__t_dt",
    "Generic projectable time scaling",
    lambda t, x: jnp.concatenate(
        [jnp.array([time_basis(t)], dtype=jnp.float32), zero_xi()]
    ),
)

add_basis(
    "time_quadratic__t2_dt",
    "Generic projectable time quadratic",
    lambda t, x: jnp.concatenate(
        [jnp.array([time_basis(t) ** 2], dtype=jnp.float32), zero_xi()]
    ),
)


# ============================================================
# Group B: coordinate translations ∂x_i
# ============================================================

for i in range(D):
    add_basis(
        f"translation__dx{i+1}",
        "Generic coordinate translation",
        lambda t, x, i=i: jnp.concatenate(
            [jnp.array([0.0], dtype=jnp.float32), e_i(i)]
        ),
    )


# ============================================================
# Group C: coordinate scalings x_i ∂x_i
# ============================================================

for i in range(D):
    add_basis(
        f"scaling__x{i+1}_dx{i+1}",
        "Generic coordinate scaling",
        lambda t, x, i=i: jnp.concatenate(
            [
                jnp.array([0.0], dtype=jnp.float32),
                jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[i]),
            ]
        ),
    )


# ============================================================
# Group D: global Brownian/Gaeta lifted scaling
# ============================================================

add_basis(
    "global_brownian_scaling__2t_dt_plus_x_dot_dx",
    "50D lift of Brownian/Gaeta scaling",
    lambda t, x: jnp.concatenate(
        [jnp.array([2.0 * time_basis(t)], dtype=jnp.float32), x]
    ),
)


# ============================================================
# Group E: Gaeta-style exponential linear-drift basis
# ============================================================

if CONFIG.get("BASIS_ADD_EXPONENTIAL", False):
    for mu in CONFIG["BASIS_LINEAR_MU_LIST"]:
        mu = float(mu)

        if abs(mu) < 1e-12:
            continue

        # e^{mu t/T} ∂x_i
        for i in range(D):
            add_basis(
                f"exp_translation__mu_{mu:+.2f}__e_mut_dx{i+1}",
                "Gaeta-style exponential translation e^{mu t} ∂x_i",
                lambda t, x, mu=mu, i=i: jnp.concatenate(
                    [
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.zeros((D,), dtype=jnp.float32).at[i].set(
                            safe_exp(mu * time_basis(t))
                        ),
                    ]
                ),
            )

        # e^{2mu t/T}(∂t + mu x·∇x)
        add_basis(
            f"exp_projective__mu_{mu:+.2f}__e_2mut_dt_plus_mu_x_dot_dx",
            "Gaeta-style projective linear-drift generator",
            lambda t, x, mu=mu: jnp.concatenate(
                [
                    jnp.array(
                        [safe_exp(2.0 * mu * time_basis(t))],
                        dtype=jnp.float32,
                    ),
                    mu * safe_exp(2.0 * mu * time_basis(t)) * x,
                ]
            ),
        )


# ============================================================
# Group F: low-rank rotations and shears
# ============================================================

LOW_RANK_DIRECTIONS = make_low_rank_directions(CONFIG["BASIS_LOW_RANK_K"])
K_LOW = len(LOW_RANK_DIRECTIONS)

print(f"Using {K_LOW} fixed low-rank basis directions.")


# ----------------------------
# Low-rank rotations:
#   (v_b·x)v_a - (v_a·x)v_b
# ----------------------------
if CONFIG.get("BASIS_ADD_LOW_RANK_ROTATIONS", False):
    for a in range(K_LOW):
        for b in range(a + 1, K_LOW):
            va = LOW_RANK_DIRECTIONS[a]
            vb = LOW_RANK_DIRECTIONS[b]

            add_basis(
                f"lowrank_rotation__v{a+1}_v{b+1}",
                "Low-rank antisymmetric rotation generator",
                lambda t, x, va=va, vb=vb: jnp.concatenate(
                    [
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.dot(vb, x) * va - jnp.dot(va, x) * vb,
                    ]
                ),
            )


# ----------------------------
# Low-rank adjacent shears:
#   (v_b·x)v_a
#
# We use only adjacent directions and both orientations:
#   v_{a+1} -> v_a and v_a -> v_{a+1}
#
# This adds 2(K-1) shears instead of all K(K-1).
# ----------------------------
if CONFIG.get("BASIS_ADD_LOW_RANK_SHEARS", False):
    for a in range(K_LOW - 1):
        va = LOW_RANK_DIRECTIONS[a]
        vb = LOW_RANK_DIRECTIONS[a + 1]

        add_basis(
            f"lowrank_shear__v{a+2}_to_v{a+1}",
            "Low-rank adjacent shear generator",
            lambda t, x, va=va, vb=vb: jnp.concatenate(
                [
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(vb, x) * va,
                ]
            ),
        )

        add_basis(
            f"lowrank_shear__v{a+1}_to_v{a+2}",
            "Low-rank adjacent shear generator",
            lambda t, x, va=va, vb=vb: jnp.concatenate(
                [
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(va, x) * vb,
                ]
            ),
        )


# ============================================================
# Finalize basis library
# ============================================================

BASIS_NAMES = [b["name"] for b in BASIS_LIBRARY]
BASIS_SOURCES = [b["source"] for b in BASIS_LIBRARY]
BASIS_FNS = tuple(b["fn"] for b in BASIS_LIBRARY)
N_BASIS = len(BASIS_FNS)

print(f"\nNumber of 50D basis functions: {N_BASIS}")
print(f"BASIS_MODE: {CONFIG['BASIS_MODE']}")

n_minimal = 3 + D + D + 1
n_exp = 0
if CONFIG.get("BASIS_ADD_EXPONENTIAL", False):
    n_exp = len([mu for mu in CONFIG["BASIS_LINEAR_MU_LIST"] if abs(mu) > 1e-12]) * (D + 1)

n_rot = 0
if CONFIG.get("BASIS_ADD_LOW_RANK_ROTATIONS", False):
    n_rot = K_LOW * (K_LOW - 1) // 2

n_shear = 0
if CONFIG.get("BASIS_ADD_LOW_RANK_SHEARS", False):
    n_shear = 2 * (K_LOW - 1)

print("\nBasis count breakdown:")
print(f"  minimal time/translation/scaling/global : {n_minimal}")
print(f"  exponential Gaeta terms                 : {n_exp}")
print(f"  low-rank rotations                      : {n_rot}")
print(f"  low-rank shears                         : {n_shear}")
print(f"  total                                   : {N_BASIS}")

for i, (nm, src) in enumerate(zip(BASIS_NAMES, BASIS_SOURCES)):
    print(f"{i:03d} | {nm} | {src}")


def eval_basis_one(t, x):
    """
    Returns basis values at one point.

    Output shape:
        (R, 51)
    """
    return jnp.stack([fn(t, x) for fn in BASIS_FNS], axis=0)


def eval_generators_point(W, z):
    """
    Evaluate learned generators at one point.

    W shape:
        (m, R)

    z shape:
        (51,) = [t, x1, ..., x50]

    Returns:
        X shape (m, 51), with X[a] = [tau_a, xi_a1, ..., xi_a50].
    """
    t = z[0]
    x = z[1:]
    B = eval_basis_one(t, x)
    return W @ B


def eval_generators_batch(W, tx):
    """
    tx shape:
        (S, 51)

    Returns:
        X shape (S, m, 51)
    """
    return jax.vmap(lambda z: eval_generators_point(W, z))(tx)


def _x_box_to_arrays(x_box):
    """
    Allows x_box to be either:
        (low_scalar, high_scalar)
    or:
        (low_vector_shape_D, high_vector_shape_D)
    """
    low, high = x_box

    low = jnp.asarray(low, dtype=jnp.float32)
    high = jnp.asarray(high, dtype=jnp.float32)

    if low.ndim == 0:
        low = jnp.full((D,), low, dtype=jnp.float32)

    if high.ndim == 0:
        high = jnp.full((D,), high, dtype=jnp.float32)

    if low.shape != (D,) or high.shape != (D,):
        raise ValueError(
            f"x_box low/high must be scalar or shape ({D},). "
            f"Got {low.shape}, {high.shape}."
        )

    return low, high


def sample_tx_box(key, n_points, t_box, x_box):
    """
    Sample collocation points in:
        [t_min,t_max] x x_box^50

    Output:
        tx shape (n_points, 51)
    """
    key_t, key_x = jax.random.split(key)

    t = jax.random.uniform(
        key_t,
        (n_points, 1),
        minval=float(t_box[0]),
        maxval=float(t_box[1]),
    )

    x_low, x_high = _x_box_to_arrays(x_box)

    u = jax.random.uniform(key_x, (n_points, D))
    x = x_low[None, :] + u * (x_high[None, :] - x_low[None, :])

    return jnp.concatenate([t, x], axis=1)

def sample_tx_data_jitter(key, dataset, n_points, jitter_std=0.05):
    """
    Sample symmetry collocation points near the observed trajectory manifold.
    """
    key_idx, key_noise = jax.random.split(key)

    t0, x0, _, _ = dataset
    n = t0.shape[0]

    idx = jax.random.randint(key_idx, (n_points,), 0, n)

    t = t0[idx]
    x = x0[idx]

    noise = jitter_std * jax.random.normal(key_noise, x.shape)
    x_jitter = x + noise

    return jnp.concatenate([t[:, None], x_jitter], axis=1)


key, subkey = jax.random.split(key)
#W = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
    #subkey,
    #(CONFIG["N_GENERATORS"], N_BASIS),
#)

#print("\nInitial W shape:", W.shape)

## Cell 4B — Deterministic E3 sampling utilities

In [ ]:
# ============================================================
# Cell 4B — Deterministic E3 sampling utilities
# ============================================================

def e3_epoch_key(base_seed, epoch):
    """Same epoch-specific sample key in every alternating cycle."""
    base = jax.random.PRNGKey(int(base_seed))
    return jax.random.fold_in(base, int(epoch))


def e3_cycle_key(base_seed, cycle_index):
    """Cycle-specific key, used only for fresh W initialization."""
    base = jax.random.PRNGKey(int(base_seed))
    return jax.random.fold_in(base, int(cycle_index))


def initialize_W_for_cycle(cycle_index):
    if CONFIG["E3_DISCOVERY_INIT"] != "fresh":
        raise ValueError(
            "Primary E3 protocol requires E3_DISCOVERY_INIT='fresh'."
        )
    init_key = e3_cycle_key(CONFIG["E3_DISCOVERY_INIT_SEED"], cycle_index)
    return CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
        init_key,
        (CONFIG["N_GENERATORS"], N_BASIS),
        dtype=jnp.float32,
    )

## Cell 5 — Determining equations and reusable Stage II

In [ ]:
# ============================================================
# Cell 5 — Determining equations and reusable Stage II
# ============================================================

def det_residuals_point(sde_params, W, z):
    """Projectable Ito determining residuals for diagonal diffusion."""
    f = f_vec_single(sde_params, z)
    sigma = sigma_vec_single(sde_params, z)

    J_f = jax.jacfwd(lambda zz: f_vec_single(sde_params, zz))(z)
    J_sigma = jax.jacfwd(lambda zz: sigma_vec_single(sde_params, zz))(z)

    f_t, f_x = J_f[:, 0], J_f[:, 1:]
    sigma_t, sigma_x = J_sigma[:, 0], J_sigma[:, 1:]

    gen_fun = lambda zz: eval_generators_point(W, zz)
    X = gen_fun(z)
    J = jax.jacfwd(gen_fun)(z)

    tau = X[:, 0]
    xi = X[:, 1:]
    tau_t = J[:, 0, 0]
    tau_x = J[:, 0, 1:]
    xi_t = J[:, 1:, 0]
    xi_x = J[:, 1:, 1:]

    # Every basis element in the supplied 50D library is at most linear in x,
    # so the Ito Hessian term 0.5*a^{jk}*d_{x_j x_k} xi vanishes.
    adv_xi = jnp.einsum("j,aij->ai", f, xi_x)
    xi_grad_f = jnp.einsum("aj,ij->ai", xi, f_x)
    time_terms = tau[:, None] * f_t[None, :] + tau_t[:, None] * f[None, :]
    R_f = xi_t + adv_xi - xi_grad_f - time_terms

    eye = jnp.eye(D, dtype=z.dtype)
    first_term = jnp.einsum("p,aip->aip", sigma, xi_x)
    diagonal_correction = (
        jnp.einsum("aj,ij->ai", xi, sigma_x)
        + tau[:, None] * sigma_t[None, :]
        + 0.5 * tau_t[:, None] * sigma[None, :]
    )
    R_sigma = first_term - diagonal_correction[:, :, None] * eye[None, :, :]
    return R_f, R_sigma, tau_x


def determining_loss(sde_params, W, tx):
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, W, z)
    )(tx)
    det = jnp.mean(R_f**2) + jnp.mean(R_sigma**2)
    projectable = jnp.mean(tau_x**2)
    return det, projectable


def generator_jacobians_batch(W, tx):
    def one(z):
        gen_fun = lambda zz: eval_generators_point(W, zz)
        return gen_fun(z), jax.jacfwd(gen_fun)(z)
    return jax.vmap(one)(tx)


def lie_closure_loss_global(W, tx):
    m = W.shape[0]
    if m < 2:
        return jnp.array(0.0, dtype=jnp.float32)

    X, J = generator_jacobians_batch(W, tx)
    S = tx.shape[0]
    idx_i, idx_j = jnp.triu_indices(m, k=1)
    P = idx_i.shape[0]

    Xi, Xj = X[:, idx_i, :], X[:, idx_j, :]
    Ji, Jj = J[:, idx_i, :, :], J[:, idx_j, :, :]
    brackets = (
        jnp.einsum("spc,spqc->spq", Xi, Jj)
        - jnp.einsum("spc,spqc->spq", Xj, Ji)
    )

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T
    B = jnp.transpose(brackets, (1, 0, 2)).reshape(P, -1).T
    G = V.T @ V
    reg = (CONFIG["CLOSURE_RCOND"] ** 2) * jnp.eye(m, dtype=G.dtype)
    C = jnp.linalg.solve(G + reg, V.T @ B)
    return jnp.mean((B - V @ C) ** 2)


def gram_independence_loss(W, tx):
    X = eval_generators_batch(W, tx)
    S, m, _ = X.shape
    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T
    G = (V.T @ V) / (Q * S)
    I = jnp.eye(m, dtype=G.dtype)
    loss_ind = jnp.mean((G - I) ** 2)
    diag = jnp.diag(G)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, CONFIG["MIN_GENERATOR_NORM"] - diag) ** 2
    )
    return loss_ind, norm_hinge, G


def sym_discovery_objective(W, sde_params_frozen, tx):
    det, proj = determining_loss(sde_params_frozen, W, tx)
    close = lie_closure_loss_global(W, tx)
    ind, norm_hinge, _ = gram_independence_loss(W, tx)
    total = (
        CONFIG["LAMBDA_DET"] * det
        + CONFIG["LAMBDA_CLOSE"] * close
        + CONFIG["LAMBDA_IND"] * ind
        + CONFIG["LAMBDA_PROJECTABLE"] * proj
        + CONFIG["LAMBDA_NORM_HINGE"] * norm_hinge
    )
    return total, {
        "total": total,
        "det": det,
        "close": close,
        "ind": ind,
        "projectable": proj,
        "norm_hinge": norm_hinge,
    }


sym_value_and_grad = jax.value_and_grad(
    lambda W, theta, tx: sym_discovery_objective(W, theta, tx)[0]
)


@jax.jit
def sym_train_step(W, opt_state, theta_frozen, tx):
    _, grads = sym_value_and_grad(W, theta_frozen, tx)
    W, opt_state = adam_update(
        W, grads, opt_state, CONFIG["SYM_DISCOVERY_LR"]
    )
    _, aux = sym_discovery_objective(W, theta_frozen, tx)
    return W, opt_state, aux


def discover_symmetries(theta_frozen, cycle_index):
    """
    Freshly discover W_k from frozen theta_{k-1}.

    The sampled jitter cloud at epoch e is identical for every cycle; only the
    cycle-specific fresh W initialization differs.
    """
    W = initialize_W_for_cycle(cycle_index)
    opt_state = adam_init(W)
    history = []

    print("\n" + "=" * 84)
    print(f"E3 CYCLE {cycle_index}: STAGE II DISCOVERY")
    print("=" * 84)

    t0 = time.perf_counter()
    for epoch in range(1, CONFIG["SYM_DISCOVERY_EPOCHS"] + 1):
        tx_key = e3_epoch_key(CONFIG["E3_DISCOVERY_SAMPLE_SEED"], epoch)
        tx_disc = sample_tx_data_jitter(
            tx_key,
            train_data,
            CONFIG["SYM_DISCOVERY_POINTS"],
            jitter_std=CONFIG["SYM_JITTER_STD"],
        )
        W, opt_state, aux = sym_train_step(
            W, opt_state, theta_frozen, tx_disc
        )

        if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
            aux_np = {name: float(value) for name, value in aux.items()}
            aux_np["epoch"] = epoch
            history.append(aux_np)
            print(
                f"[Cycle {cycle_index} | discovery] epoch {epoch:5d} | "
                f"total {aux_np['total']: .4e} | "
                f"det {aux_np['det']: .4e} | "
                f"close {aux_np['close']: .4e} | "
                f"ind {aux_np['ind']: .4e} | "
                f"proj {aux_np['projectable']: .4e}"
            )

    W.block_until_ready()
    seconds = time.perf_counter() - t0
    print(f"Cycle {cycle_index} discovery time: {seconds:.2f} sec")
    return jnp.array(W), history, seconds


## Cell 6A — Reusable frozen-W Stage III

In [ ]:
# ============================================================
# Cell 6A — Reusable frozen-W Stage III
# ============================================================

def stage_f_objective(theta, W_fixed, batch, tx_extra):
    data_loss = data_nll_loss(theta, batch)
    t_batch, x_batch, _, _ = batch
    n_data_sym = min(CONFIG["STAGE_F_N_DATA_SYM_POINTS"], t_batch.shape[0])
    tx_data = jnp.concatenate(
        [t_batch[:n_data_sym, None], x_batch[:n_data_sym]], axis=1
    )
    tx_sym = jnp.concatenate([tx_data, tx_extra], axis=0)
    det, proj = determining_loss(theta, W_fixed, tx_sym)
    total = (
        data_loss
        + CONFIG["STAGE_F_LAMBDA_SYM_DET"] * det
        + CONFIG["STAGE_F_LAMBDA_PROJECTABLE"] * proj
    )
    return total, {
        "total": total,
        "data": data_loss,
        "sym_det": det,
        "projectable": proj,
    }


stage_f_value_and_grad = jax.value_and_grad(
    lambda theta, W_fixed, batch, tx_extra: stage_f_objective(
        theta, W_fixed, batch, tx_extra
    )[0]
)


@jax.jit
def stage_f_train_step(theta, opt_state, W_fixed, batch, tx_extra):
    _, grads = stage_f_value_and_grad(theta, W_fixed, batch, tx_extra)
    theta, opt_state = adam_update(
        theta, grads, opt_state, CONFIG["STAGE_F_LR"]
    )
    _, aux = stage_f_objective(theta, W_fixed, batch, tx_extra)
    return theta, opt_state, aux


def refine_surrogate(theta_start, W_fixed, cycle_index):
    """
    Refine theta_{k-1} into theta_k with W_k frozen.

    Adam is reset every cycle. The data-batch and extra-collocation keys at
    epoch e are identical across cycles.
    """
    theta = tree_copy(theta_start)
    opt_state = adam_init(theta)
    history = []

    print("\n" + "=" * 84)
    print(f"E3 CYCLE {cycle_index}: STAGE III REFINEMENT")
    print("=" * 84)

    t0 = time.perf_counter()
    for epoch in range(1, CONFIG["STAGE_F_EPOCHS"] + 1):
        batch_key = e3_epoch_key(CONFIG["E3_REFINEMENT_BATCH_SEED"], epoch)
        extra_key = e3_epoch_key(CONFIG["E3_REFINEMENT_EXTRA_SEED"], epoch)

        batch = sample_batch(
            batch_key, train_data, CONFIG["STAGE_F_BATCH_SIZE"]
        )
        tx_extra = sample_tx_data_jitter(
            extra_key,
            train_data,
            CONFIG["STAGE_F_N_EXTRA_SYM_POINTS"],
            jitter_std=CONFIG["SYM_JITTER_STD"],
        )

        theta, opt_state, aux = stage_f_train_step(
            theta, opt_state, W_fixed, batch, tx_extra
        )

        if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
            aux_np = {name: float(value) for name, value in aux.items()}
            aux_np["epoch"] = epoch
            history.append(aux_np)
            train_nll = float(data_nll_loss(theta, train_data))
            test_nll = float(data_nll_loss(theta, test_data))
            print(
                f"[Cycle {cycle_index} | refinement] epoch {epoch:5d} | "
                f"total {aux_np['total']: .6f} | "
                f"data {aux_np['data']: .6f} | "
                f"sym {aux_np['sym_det']: .4e} | "
                f"train NLL {train_nll: .6f} | "
                f"test NLL {test_nll: .6f}"
            )

    tree_block_until_ready(theta)
    seconds = time.perf_counter() - t0
    print(f"Cycle {cycle_index} refinement time: {seconds:.2f} sec")
    return tree_copy(theta), history, seconds


# ============================================================
# Cell 6B — Fixed evaluation clouds and E3 diagnostics
# ============================================================

def mse(a, b):
    return jnp.mean((a - b) ** 2)


def rmse(a, b):
    return jnp.sqrt(mse(a, b))


def mae(a, b):
    return jnp.mean(jnp.abs(a - b))


def relative_l2_error(pred, truth, eps=1e-12):
    return jnp.sqrt(
        jnp.sum((pred - truth) ** 2) / (jnp.sum(truth**2) + eps)
    )


def batched_surrogate_values(params, t_values, x_values):
    return jax.vmap(
        lambda tt, xx: surrogate_f_sigma(params, tt, xx)
    )(t_values, x_values)


def make_fixed_e3_evaluation_sets():
    # Primary support-aware held-out cloud.
    primary_key = jax.random.PRNGKey(CONFIG["E3_EVAL_SEED"])
    tx_primary = sample_tx_data_jitter(
        primary_key,
        test_data,
        CONFIG["E3_PRIMARY_EVAL_POINTS"],
        jitter_std=CONFIG["E3_PRIMARY_EVAL_JITTER_STD"],
    )

    # Secondary global box cloud.
    box_key = jax.random.fold_in(primary_key, 1)
    tx_box = sample_tx_box(
        box_key,
        CONFIG["E3_BOX_EVAL_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
        CONFIG["STAGE_F_EXTRA_X_BOX"],
    )

    # Held-out determining-residual cloud, never used in optimization.
    residual_key = jax.random.PRNGKey(CONFIG["E3_RESIDUAL_SEED"])
    tx_residual = sample_tx_data_jitter(
        residual_key,
        test_data,
        CONFIG["E3_RESIDUAL_EVAL_POINTS"],
        jitter_std=CONFIG["E3_PRIMARY_EVAL_JITTER_STD"],
    )

    # Fixed function-space cloud for generator principal angles.
    subspace_key = jax.random.PRNGKey(CONFIG["E3_SUBSPACE_SEED"])
    tx_subspace = sample_tx_data_jitter(
        subspace_key,
        test_data,
        CONFIG["E3_SUBSPACE_EVAL_POINTS"],
        jitter_std=CONFIG["E3_PRIMARY_EVAL_JITTER_STD"],
    )

    return {
        "primary": tx_primary,
        "box": tx_box,
        "residual": tx_residual,
        "subspace": tx_subspace,
    }


E3_EVAL = make_fixed_e3_evaluation_sets()


def coefficient_metrics_on_tx(params, tx):
    t, x = tx[:, 0], tx[:, 1:]
    f_true = jax.vmap(lambda tt, xx: true_drift(tt, xx))(t, x)
    s_true = jax.vmap(lambda tt, xx: true_sigma(tt, xx))(t, x)
    f_pred, s_pred = batched_surrogate_values(params, t, x)
    return {
        "drift_mse": float(mse(f_pred, f_true)),
        "drift_rmse": float(rmse(f_pred, f_true)),
        "drift_mae": float(mae(f_pred, f_true)),
        "drift_rel_l2": float(relative_l2_error(f_pred, f_true)),
        "sigma_mse": float(mse(s_pred, s_true)),
        "sigma_rmse": float(rmse(s_pred, s_true)),
        "sigma_mae": float(mae(s_pred, s_true)),
        "sigma_rel_l2": float(relative_l2_error(s_pred, s_true)),
    }


def one_step_prediction_diagnostics(params, dataset):
    t0, x0, dt, dx = dataset
    f, s = jax.vmap(
        lambda tt, xx: surrogate_f_sigma(params, tt, xx)
    )(t0, x0)
    mean_pred = f * dt[:, None]
    var_pred = s**2 * dt[:, None] + 1e-8
    return {
        "increment_rmse": float(rmse(mean_pred, dx)),
        "increment_mae": float(mae(mean_pred, dx)),
        "standardized_sq_residual_mean": float(
            jnp.mean((dx - mean_pred) ** 2 / var_pred)
        ),
        "avg_pred_step_std": float(jnp.mean(jnp.sqrt(var_pred))),
        "avg_abs_dx": float(jnp.mean(jnp.abs(dx))),
    }


def function_space_basis(W, tx, rcond=None):
    """
    Orthonormal basis for span{X_1,...,X_m} evaluated on a fixed cloud.

    Returns U_r, singular values, and numerical rank. Columns of U_r span the
    learned generator subspace in sampled function space.
    """
    if rcond is None:
        rcond = CONFIG["E3_SUBSPACE_RCOND"]

    X = np.asarray(eval_generators_batch(W, tx), dtype=np.float64)
    # X: (S,m,Q) -> each generator becomes one flattened function-space column.
    V = np.transpose(X, (1, 0, 2)).reshape(X.shape[1], -1).T
    U, singular_values, _ = np.linalg.svd(V, full_matrices=False)

    if singular_values.size == 0 or singular_values[0] <= 0.0:
        return np.zeros((V.shape[0], 0)), singular_values, 0

    rank = int(np.sum(singular_values > rcond * singular_values[0]))
    return U[:, :rank], singular_values, rank


def principal_angle_metrics(W_previous, W_current, tx):
    U_prev, s_prev, rank_prev = function_space_basis(W_previous, tx)
    U_curr, s_curr, rank_curr = function_space_basis(W_current, tx)

    if rank_prev == 0 or rank_curr == 0:
        return {
            "angle_min_deg": np.nan,
            "angle_mean_deg": np.nan,
            "angle_median_deg": np.nan,
            "angle_max_deg": np.nan,
            "canonical_corr_max": np.nan,
            "canonical_corr_min": np.nan,
            "subspace_rank_previous": rank_prev,
            "subspace_rank_current": rank_curr,
        }

    canonical_corr = np.linalg.svd(U_prev.T @ U_curr, compute_uv=False)
    canonical_corr = np.clip(canonical_corr, 0.0, 1.0)
    angles_deg = np.degrees(np.arccos(canonical_corr))

    return {
        "angle_min_deg": float(np.min(angles_deg)),
        "angle_mean_deg": float(np.mean(angles_deg)),
        "angle_median_deg": float(np.median(angles_deg)),
        "angle_max_deg": float(np.max(angles_deg)),
        "canonical_corr_max": float(np.max(canonical_corr)),
        "canonical_corr_min": float(np.min(canonical_corr)),
        "subspace_rank_previous": rank_prev,
        "subspace_rank_current": rank_curr,
    }


def generator_rank_metrics(W, tx):
    _, singular_values, rank = function_space_basis(W, tx)
    return {
        "subspace_rank_current": rank,
        "subspace_singular_max": (
            float(singular_values[0]) if singular_values.size else np.nan
        ),
        "subspace_singular_min_retained": (
            float(singular_values[rank - 1]) if rank > 0 else np.nan
        ),
    }


def evaluate_theta_record(
    theta,
    cycle_index,
    stage_label,
    cumulative_seconds,
    W_current=None,
    W_previous=None,
    theta_pre_refine=None,
    discovery_seconds=0.0,
    refinement_seconds=0.0,
):
    primary = coefficient_metrics_on_tx(theta, E3_EVAL["primary"])
    box = coefficient_metrics_on_tx(theta, E3_EVAL["box"])
    test_step = one_step_prediction_diagnostics(theta, test_data)

    record = {
        "cycle": int(cycle_index),
        "stage_label": stage_label,
        "n_generators": int(CONFIG["N_GENERATORS"]),
        **primary,
        "box_drift_mse": box["drift_mse"],
        "box_drift_rmse": box["drift_rmse"],
        "box_sigma_mse": box["sigma_mse"],
        "box_sigma_rmse": box["sigma_rmse"],
        "train_nll": float(data_nll_loss(theta, train_data)),
        "test_nll": float(data_nll_loss(theta, test_data)),
        "test_increment_rmse": test_step["increment_rmse"],
        "test_increment_mae": test_step["increment_mae"],
        "test_standardized_sq_residual": test_step[
            "standardized_sq_residual_mean"
        ],
        "discovery_seconds": float(discovery_seconds),
        "refinement_seconds": float(refinement_seconds),
        "cumulative_seconds": float(cumulative_seconds),
        "det_pre_refine": np.nan,
        "det_post_refine": np.nan,
        "projectability_pre": np.nan,
        "projectability_post": np.nan,
        "angle_min_deg": np.nan,
        "angle_mean_deg": np.nan,
        "angle_median_deg": np.nan,
        "angle_max_deg": np.nan,
        "canonical_corr_max": np.nan,
        "canonical_corr_min": np.nan,
        "subspace_rank_previous": np.nan,
        "subspace_rank_current": np.nan,
    }

    if W_current is not None:
        if theta_pre_refine is None:
            raise ValueError("theta_pre_refine is required when W_current is given.")

        det_pre, proj_pre = determining_loss(
            theta_pre_refine, W_current, E3_EVAL["residual"]
        )
        det_post, proj_post = determining_loss(
            theta, W_current, E3_EVAL["residual"]
        )
        record.update({
            "det_pre_refine": float(det_pre),
            "det_post_refine": float(det_post),
            "projectability_pre": float(proj_pre),
            "projectability_post": float(proj_post),
        })

        if W_previous is None:
            record.update(generator_rank_metrics(W_current, E3_EVAL["subspace"]))
        else:
            record.update(principal_angle_metrics(
                W_previous, W_current, E3_EVAL["subspace"]
            ))

    return record


# ============================================================
# Cell 6C — Execute theta0 -> W1 -> theta1 -> W2 -> theta2 -> W3 -> theta3
# ============================================================

theta_by_cycle = [tree_copy(theta_0)]
W_by_cycle = []
discovery_histories = []
refinement_histories = []
cycle_records = []

cycle_records.append(
    evaluate_theta_record(
        theta=theta_by_cycle[0],
        cycle_index=0,
        stage_label="Stage I / theta_0",
        cumulative_seconds=stage1_seconds,
    )
)

cumulative_seconds = float(stage1_seconds)

for cycle_index in range(1, CONFIG["E3_N_CYCLES"] + 1):
    theta_previous = theta_by_cycle[-1]

    W_k, disc_history_k, discovery_seconds_k = discover_symmetries(
        theta_frozen=theta_previous,
        cycle_index=cycle_index,
    )

    theta_k, refine_history_k, refinement_seconds_k = refine_surrogate(
        theta_start=theta_previous,
        W_fixed=W_k,
        cycle_index=cycle_index,
    )

    cumulative_seconds += discovery_seconds_k + refinement_seconds_k

    W_previous = W_by_cycle[-1] if W_by_cycle else None
    record_k = evaluate_theta_record(
        theta=theta_k,
        cycle_index=cycle_index,
        stage_label=(
            "submitted pass / theta_1"
            if cycle_index == 1
            else f"iterative pass / theta_{cycle_index}"
        ),
        cumulative_seconds=cumulative_seconds,
        W_current=W_k,
        W_previous=W_previous,
        theta_pre_refine=theta_previous,
        discovery_seconds=discovery_seconds_k,
        refinement_seconds=refinement_seconds_k,
    )

    W_by_cycle.append(jnp.array(W_k))
    theta_by_cycle.append(tree_copy(theta_k))
    discovery_histories.append(disc_history_k)
    refinement_histories.append(refine_history_k)
    cycle_records.append(record_k)

    print("\nCycle summary")
    print(f"  cycle                : {cycle_index}")
    print(f"  primary drift MSE    : {record_k['drift_mse']:.6e}")
    print(f"  primary sigma MSE    : {record_k['sigma_mse']:.6e}")
    print(f"  held-out test NLL    : {record_k['test_nll']:.6e}")
    print(f"  det pre refinement   : {record_k['det_pre_refine']:.6e}")
    print(f"  det post refinement  : {record_k['det_post_refine']:.6e}")
    if cycle_index >= 2:
        print(f"  mean principal angle : {record_k['angle_mean_deg']:.3f} deg")
        print(f"  max principal angle  : {record_k['angle_max_deg']:.3f} deg")

# Compatibility aliases for any optional downstream plotting.
sde_params_refined = theta_by_cycle[-1]
W = W_by_cycle[-1]

print("\nCompleted E3 alternating loop.")
print("theta checkpoints:", len(theta_by_cycle))
print("W checkpoints    :", len(W_by_cycle))

## Cell 7 — Final automatic E3 numerical report

In [ ]:
# ============================================================
# Cell 7 — Final automatic E3 numerical report
# ============================================================

def _finite(x):
    try:
        return bool(np.isfinite(float(x)))
    except (TypeError, ValueError):
        return False


def _as_float(record, key, default=np.nan):
    try:
        return float(record.get(key, default))
    except (TypeError, ValueError):
        return float(default)


def _fmt(x, fmt=".6e", missing="NA"):
    return format(float(x), fmt) if _finite(x) else missing


def _pct_change(new, old, eps=1e-12):
    if not (_finite(new) and _finite(old)):
        return np.nan
    return 100.0 * (float(new) - float(old)) / (abs(float(old)) + eps)


def _pct_reduction(new, old, eps=1e-12):
    value = _pct_change(new, old, eps)
    return -value if _finite(value) else np.nan


def _relative_change_abs(new, old, eps=1e-12):
    if not (_finite(new) and _finite(old)):
        return np.nan
    return abs(float(new) - float(old)) / (abs(float(old)) + eps)

def _nonincreasing(values, rtol=1e-3, atol=1e-12):
    """
    Return True when no successive value increases beyond the declared
    absolute-plus-relative numerical tolerance.

    Unlike the previous implementation, the relative tolerance is scaled by
    the actual magnitude of the preceding metric rather than by a floor of
    1.0. This is essential for coefficient MSEs that are much smaller than 1.
    """
    values = np.asarray(values, dtype=float)

    if len(values) < 2 or not np.all(np.isfinite(values)):
        return False

    previous = values[:-1]
    current = values[1:]

    allowed_increase = atol + rtol * np.abs(previous)

    return bool(
        np.all(current <= previous + allowed_increase)
    )


records = sorted(cycle_records, key=lambda r: int(r["cycle"]))
actual_cycles = [int(r["cycle"]) for r in records]
if actual_cycles != list(range(len(records))):
    raise RuntimeError(f"Nonconsecutive E3 cycles: {actual_cycles}")

cycles = np.asarray(actual_cycles)
drift_mse = np.asarray([_as_float(r, "drift_mse") for r in records])
sigma_mse = np.asarray([_as_float(r, "sigma_mse") for r in records])
box_drift_mse = np.asarray([_as_float(r, "box_drift_mse") for r in records])
box_sigma_mse = np.asarray([_as_float(r, "box_sigma_mse") for r in records])
test_nll = np.asarray([_as_float(r, "test_nll") for r in records])
angle_mean = np.asarray([_as_float(r, "angle_mean_deg") for r in records])
angle_max = np.asarray([_as_float(r, "angle_max_deg") for r in records])

print("\n" + "=" * 118)
print("E3 FINAL REPORT — ITERATED DISCOVERY-REFINEMENT")
print("=" * 118)
print(f"System                         : {CONFIG['TRUE_SDE_NAME']}")
print(f"Description                    : {ACTIVE_SDE['description']}")
print(f"State dimension                : {D}")
print(f"Training trajectories          : {CONFIG['N_TRAJECTORIES']}")
print(f"Alternating cycles completed   : {CONFIG['E3_N_CYCLES']}")
print(f"Generators per cycle           : {CONFIG['N_GENERATORS']}")
print(f"Discovery init                 : {CONFIG['E3_DISCOVERY_INIT']}")
print(f"Primary coefficient cloud      : {CONFIG['E3_PRIMARY_EVAL_DISTRIBUTION']}")
print("Cycle 1 is the submitted single pass; cycles 2-3 are optional iterative extensions.")

print("\n[1] PRIMARY HELD-OUT SUPPORT-AWARE METRICS")
header = (
    f"{'cycle':>5}  {'stage':<28}  {'drift MSE':>13}  {'sigma MSE':>13}  "
    f"{'test NLL':>13}  {'det pre':>13}  {'det post':>13}  "
    f"{'mean angle':>11}  {'max angle':>11}"
)
print(header)
print("-" * len(header))
for r in records:
    print(
        f"{int(r['cycle']):5d}  {str(r['stage_label'])[:28]:<28}  "
        f"{_fmt(r['drift_mse']):>13}  {_fmt(r['sigma_mse']):>13}  "
        f"{_fmt(r['test_nll']):>13}  {_fmt(r['det_pre_refine']):>13}  "
        f"{_fmt(r['det_post_refine']):>13}  "
        f"{_fmt(r['angle_mean_deg'], '.3f'):>11}  "
        f"{_fmt(r['angle_max_deg'], '.3f'):>11}"
    )

print("\n[2] SECONDARY GLOBAL BOX COEFFICIENT METRICS")
header = (
    f"{'cycle':>5}  {'box drift MSE':>16}  {'box sigma MSE':>16}  "
    f"{'drift red. vs S-I':>20}  {'sigma red. vs S-I':>20}"
)
print(header)
print("-" * len(header))
for k, r in enumerate(records):
    print(
        f"{k:5d}  {_fmt(r['box_drift_mse']):>16}  "
        f"{_fmt(r['box_sigma_mse']):>16}  "
        f"{(_fmt(_pct_reduction(box_drift_mse[k], box_drift_mse[0]), '.3f') + '%'):>20}  "
        f"{(_fmt(_pct_reduction(box_sigma_mse[k], box_sigma_mse[0]), '.3f') + '%'):>20}"
    )

print("\n[3] CHANGES RELATIVE TO STAGE I AND PREVIOUS CYCLE")
header = (
    f"{'pass':>5}  {'drift red. vs S-I':>20}  {'sigma red. vs S-I':>20}  "
    f"{'NLL red. vs S-I':>18}  {'drift red. vs prev':>20}  "
    f"{'sigma red. vs prev':>20}  {'NLL red. vs prev':>18}"
)
print(header)
print("-" * len(header))
for k in range(1, len(records)):
    print(
        f"{k:5d}  "
        f"{(_fmt(_pct_reduction(drift_mse[k], drift_mse[0]), '.3f') + '%'):>20}  "
        f"{(_fmt(_pct_reduction(sigma_mse[k], sigma_mse[0]), '.3f') + '%'):>20}  "
        f"{(_fmt(_pct_reduction(test_nll[k], test_nll[0]), '.3f') + '%'):>18}  "
        f"{(_fmt(_pct_reduction(drift_mse[k], drift_mse[k-1]), '.3f') + '%'):>20}  "
        f"{(_fmt(_pct_reduction(sigma_mse[k], sigma_mse[k-1]), '.3f') + '%'):>20}  "
        f"{(_fmt(_pct_reduction(test_nll[k], test_nll[k-1]), '.3f') + '%'):>18}"
    )

print("\n[4] HELD-OUT DETERMINING-RESIDUAL CONSISTENCY")
header = (
    f"{'pass':>5}  {'pre-refine det':>16}  {'post-refine det':>16}  "
    f"{'det reduction':>16}  {'proj pre':>13}  {'proj post':>13}"
)
print(header)
print("-" * len(header))
for k, r in enumerate(records[1:], start=1):
    print(
        f"{k:5d}  {_fmt(r['det_pre_refine']):>16}  "
        f"{_fmt(r['det_post_refine']):>16}  "
        f"{(_fmt(_pct_reduction(r['det_post_refine'], r['det_pre_refine']), '.3f') + '%'):>16}  "
        f"{_fmt(r['projectability_pre']):>13}  "
        f"{_fmt(r['projectability_post']):>13}"
    )

print("\n[5] SUCCESSIVE FUNCTION-SPACE GENERATOR SUBSPACES")
header = (
    f"{'comparison':>12}  {'rank prev':>10}  {'rank curr':>10}  "
    f"{'min deg':>10}  {'mean deg':>10}  {'median deg':>12}  "
    f"{'max deg':>10}  {'min canon corr':>16}"
)
print(header)
print("-" * len(header))
for k, r in enumerate(records[1:], start=1):
    comparison = "first W" if k == 1 else f"W{k-1} -> W{k}"
    print(
        f"{comparison:>12}  {_fmt(r['subspace_rank_previous'], '.0f'):>10}  "
        f"{_fmt(r['subspace_rank_current'], '.0f'):>10}  "
        f"{_fmt(r['angle_min_deg'], '.3f'):>10}  "
        f"{_fmt(r['angle_mean_deg'], '.3f'):>10}  "
        f"{_fmt(r['angle_median_deg'], '.3f'):>12}  "
        f"{_fmt(r['angle_max_deg'], '.3f'):>10}  "
        f"{_fmt(r['canonical_corr_min'], '.6f'):>16}"
    )

print("\n[6] RUNTIME")
header = (
    f"{'cycle':>5}  {'discovery sec':>15}  {'refinement sec':>16}  "
    f"{'cycle sec':>12}  {'cumulative sec':>16}"
)
print(header)
print("-" * len(header))
for r in records:
    dsec = _as_float(r, "discovery_seconds", 0.0)
    rsec = _as_float(r, "refinement_seconds", 0.0)
    print(
        f"{int(r['cycle']):5d}  {_fmt(dsec, '.3f'):>15}  "
        f"{_fmt(rsec, '.3f'):>16}  {_fmt(dsec+rsec, '.3f'):>12}  "
        f"{_fmt(r['cumulative_seconds'], '.3f'):>16}"
    )

# ----------------------------
# Automated scientific interpretation
# ----------------------------
plateau_tol = CONFIG["E3_PLATEAU_REL_TOL"]
degrade_tol = CONFIG["E3_DEGRADE_REL_TOL"]
mean_angle_tol = CONFIG["E3_STABLE_ANGLE_MEAN_DEG"]
max_angle_tol = CONFIG["E3_STABLE_ANGLE_MAX_DEG"]

drift_monotone = _nonincreasing(drift_mse)
sigma_monotone = _nonincreasing(sigma_mse)
nll_monotone = _nonincreasing(test_nll)
first_pass_improves_both = drift_mse[1] < drift_mse[0] and sigma_mse[1] < sigma_mse[0]

if len(records) > 2:
    later_drift_degradation = np.max(
        drift_mse[2:] / (drift_mse[1] + 1e-12) - 1.0
    )
    later_sigma_degradation = np.max(
        sigma_mse[2:] / (sigma_mse[1] + 1e-12) - 1.0
    )
    final_vs_first_drift = _relative_change_abs(drift_mse[-1], drift_mse[1])
    final_vs_first_sigma = _relative_change_abs(sigma_mse[-1], sigma_mse[1])
else:
    later_drift_degradation = later_sigma_degradation = 0.0
    final_vs_first_drift = final_vs_first_sigma = 0.0

later_material_degradation = (
    later_drift_degradation > degrade_tol
    or later_sigma_degradation > degrade_tol
)
later_plateau = (
    len(records) > 2
    and final_vs_first_drift <= plateau_tol
    and final_vs_first_sigma <= plateau_tol
)

# Principal angles are defined only for comparisons W_{k-1} -> W_k,
# beginning at cycle 2. Cycle 1 has no preceding learned generator bank.
finite_successive_angles = angle_mean[2:][np.isfinite(angle_mean[2:])]
finite_successive_max_angles = angle_max[2:][np.isfinite(angle_max[2:])]


def _integer_rank_or_none(value):
    """
    Convert a stored numerical rank to int, returning None for missing or
    nonfinite entries.
    """
    if not _finite(value):
        return None
    return int(round(float(value)))


successive_rank_comparisons = []

for k in range(2, len(records)):
    rank_previous = _integer_rank_or_none(
        records[k].get("subspace_rank_previous", np.nan)
    )
    rank_current = _integer_rank_or_none(
        records[k].get("subspace_rank_current", np.nan)
    )

    comparison_available = (
        rank_previous is not None
        and rank_current is not None
    )

    ranks_equal = (
        comparison_available
        and rank_previous == rank_current
    )

    successive_rank_comparisons.append({
        "cycle": k,
        "rank_previous": rank_previous,
        "rank_current": rank_current,
        "comparison_available": comparison_available,
        "ranks_equal": ranks_equal,
    })


# Rank stability is true only when:
#   1. at least one successive learned-subspace comparison exists;
#   2. every required rank is available; and
#   3. no numerical rank is gained or lost between successive cycles.
successive_rank_stable = (
    len(successive_rank_comparisons) > 0
    and all(
        item["comparison_available"]
        for item in successive_rank_comparisons
    )
    and all(
        item["ranks_equal"]
        for item in successive_rank_comparisons
    )
)


angles_available = (
    len(finite_successive_angles) > 0
    and len(finite_successive_max_angles) > 0
    and len(finite_successive_angles)
        == len(finite_successive_max_angles)
)


subspace_stable = (
    angles_available
    and np.all(
        finite_successive_angles <= mean_angle_tol
    )
    and np.all(
        finite_successive_max_angles <= max_angle_tol
    )
    and successive_rank_stable
)

residual_improves_each_pass = all(
    r["det_post_refine"] <= r["det_pre_refine"] for r in records[1:]
)

self_consistency_warning = any(
    (records[k]["det_post_refine"] < records[k]["det_pre_refine"])
    and (
        drift_mse[k] > drift_mse[k-1] * (1.0 + degrade_tol)
        or sigma_mse[k] > sigma_mse[k-1] * (1.0 + degrade_tol)
    )
    for k in range(1, len(records))
)

if first_pass_improves_both and drift_monotone and sigma_monotone and not later_material_degradation:
    if subspace_stable:
        verdict_code = "IDEAL"
        verdict_text = (
            "Coefficient recovery improves monotonically and the learned "
            "generator subspaces stabilize across rediscovery cycles."
        )
    else:
        verdict_code = "IMPROVING_BUT_SUBSPACE_NOT_STABLE"
        verdict_text = (
            "Coefficient recovery improves monotonically, but the learned "
            "generator subspaces do not yet meet the declared stability tolerance."
        )
elif first_pass_improves_both and later_plateau and not later_material_degradation:
    verdict_code = "STRONG_SINGLE_PASS"
    verdict_text = (
        "The first pass supplies essentially all coefficient-recovery gain; "
        "later cycles plateau without material degradation. This supports the "
        "submitted frozen single-pass procedure as a stable stopping rule."
    )
elif later_material_degradation:
    verdict_code = "UNFAVORABLE_ITERATION"
    verdict_text = (
        "At least one later cycle materially degrades coefficient recovery. "
        "Do not claim safe iterability; present the frozen single pass as an "
        "intentional safeguard."
    )
else:
    verdict_code = "MIXED"
    verdict_text = (
        "The iterative extension has mixed behavior. Retain the submitted "
        "single pass as the primary method and make no convergence claim."
    )

print("\n" + "=" * 118)
print("E3 AUTOMATED VERDICT")
print("=" * 118)
print(f"Verdict code                         : {verdict_code}")
print(f"Interpretation                       : {verdict_text}")
print(f"First pass improves drift and sigma  : {first_pass_improves_both}")
print(f"Drift MSE nonincreasing              : {drift_monotone}")
print(f"Sigma MSE nonincreasing              : {sigma_monotone}")
print(f"Held-out NLL nonincreasing           : {nll_monotone}")
print(f"Residual improves within every pass  : {residual_improves_each_pass}")
print(f"Successive subspace ranks stable     : {successive_rank_stable}")
print(f"Successive generator subspace stable : {subspace_stable}")
if successive_rank_comparisons:
    print("\nSuccessive numerical-rank comparisons:")

    for item in successive_rank_comparisons:
        previous_text = (
            str(item["rank_previous"])
            if item["rank_previous"] is not None
            else "NA"
        )
        current_text = (
            str(item["rank_current"])
            if item["rank_current"] is not None
            else "NA"
        )

        print(
            f"  W_{item['cycle'] - 1} -> W_{item['cycle']}: "
            f"{previous_text} -> {current_text} | "
            f"preserved={item['ranks_equal']}"
        )
print(f"Self-consistency warning             : {self_consistency_warning}")
print(f"Plateau tolerance                    : {100*plateau_tol:.2f}%")
print(f"Degradation tolerance                : {100*degrade_tol:.2f}%")
print(f"Stable mean-angle threshold          : {mean_angle_tol:.2f} deg")
print(f"Stable max-angle threshold           : {max_angle_tol:.2f} deg")

if self_consistency_warning:
    print(
        "WARNING: a held-out determining residual decreased while coefficient "
        "MSE materially worsened. This is evidence of movement toward a more "
        "self-consistent but less coefficient-accurate symmetric surrogate."
    )

if not successive_rank_stable:
    print(
        "\nWARNING: The numerical rank of the learned generator subspace "
        "was not preserved across all successive rediscovery cycles. "
        "Small principal angles alone must therefore not be interpreted as "
        "full generator-subspace stability."
    )

# ----------------------------
# Machine-readable CSV
# ----------------------------
csv_columns = [
    "system",
    "state_dim",
    "n_generators",
    "cycle",
    "stage_label",
    "drift_mse", "sigma_mse", "drift_rmse", "sigma_rmse",
    "box_drift_mse", "box_sigma_mse",
    "train_nll", "test_nll",
    "test_increment_rmse", "test_standardized_sq_residual",
    "det_pre_refine", "det_post_refine", "det_reduction_pct",
    "projectability_pre", "projectability_post",
    "angle_min_deg", "angle_mean_deg", "angle_median_deg", "angle_max_deg",
    "canonical_corr_max", "canonical_corr_min",
    "subspace_rank_previous", "subspace_rank_current",
    "discovery_seconds", "refinement_seconds", "cumulative_seconds",
    "verdict_code",
]

csv_buffer = io.StringIO()
writer = csv.DictWriter(csv_buffer, fieldnames=csv_columns)
writer.writeheader()
for r in records:
    writer.writerow({
        "system": CONFIG["TRUE_SDE_NAME"],
        "state_dim": D,
        "n_generators": r["n_generators"],
        "cycle": r["cycle"],
        "stage_label": r["stage_label"],
        "drift_mse": r["drift_mse"],
        "sigma_mse": r["sigma_mse"],
        "drift_rmse": r["drift_rmse"],
        "sigma_rmse": r["sigma_rmse"],
        "box_drift_mse": r["box_drift_mse"],
        "box_sigma_mse": r["box_sigma_mse"],
        "train_nll": r["train_nll"],
        "test_nll": r["test_nll"],
        "test_increment_rmse": r["test_increment_rmse"],
        "test_standardized_sq_residual": r["test_standardized_sq_residual"],
        "det_pre_refine": r["det_pre_refine"],
        "det_post_refine": r["det_post_refine"],
        "det_reduction_pct": _pct_reduction(
            r["det_post_refine"], r["det_pre_refine"]
        ),
        "projectability_pre": r["projectability_pre"],
        "projectability_post": r["projectability_post"],
        "angle_min_deg": r["angle_min_deg"],
        "angle_mean_deg": r["angle_mean_deg"],
        "angle_median_deg": r["angle_median_deg"],
        "angle_max_deg": r["angle_max_deg"],
        "canonical_corr_max": r["canonical_corr_max"],
        "canonical_corr_min": r["canonical_corr_min"],
        "subspace_rank_previous": r["subspace_rank_previous"],
        "subspace_rank_current": r["subspace_rank_current"],
        "discovery_seconds": r["discovery_seconds"],
        "refinement_seconds": r["refinement_seconds"],
        "cumulative_seconds": r["cumulative_seconds"],
        "verdict_code": verdict_code,
    })

print("\n" + "=" * 118)
print("COPYABLE CSV — ONE ROW PER CYCLE")
print("=" * 118)
print(csv_buffer.getvalue())

# Save directly inside Colab/runtime as well as printing it.
csv_filename = f"E3_results_{CONFIG['TRUE_SDE_NAME']}_seed{CONFIG['SEED']}.csv"
with open(csv_filename, "w", newline="") as f:
    f.write(csv_buffer.getvalue())
print("Saved CSV:", csv_filename)

print("\nSUMMARY")
print(
    f"Stage I -> pass 1 drift MSE reduction: "
    f"{_pct_reduction(drift_mse[1], drift_mse[0]):.2f}%"
)
print(
    f"Stage I -> pass 1 diffusion MSE reduction: "
    f"{_pct_reduction(sigma_mse[1], sigma_mse[0]):.2f}%"
)
print(
    f"Stage I -> final-pass drift MSE reduction: "
    f"{_pct_reduction(drift_mse[-1], drift_mse[0]):.2f}%"
)
print(
    f"Stage I -> final-pass diffusion MSE reduction: "
    f"{_pct_reduction(sigma_mse[-1], sigma_mse[0]):.2f}%"
)
print(
    f"Pass 1 -> final-pass drift MSE change: "
    f"{_pct_change(drift_mse[-1], drift_mse[1]):+.2f}%"
)
print(
    f"Pass 1 -> final-pass diffusion MSE change: "
    f"{_pct_change(sigma_mse[-1], sigma_mse[1]):+.2f}%"
)
print(
    "Successive rank preservation: "
    f"{successive_rank_stable}"
)
if len(finite_successive_angles):
    print(
        "Mean successive generator-subspace angle over rediscoveries: "
        f"{np.mean(finite_successive_angles):.2f} deg"
    )
print("Final conclusion:", verdict_text)


## Cell 8 — Lightweight plots only (run after Cell 7)

In [ ]:
# ============================================================
# Cell 8 — Lightweight plots only (run after Cell 7)
# ============================================================

cycle_axis = np.arange(len(cycle_records))

plt.figure(figsize=(7, 4))
plt.plot(cycle_axis, [r["drift_mse"] for r in cycle_records], marker="o", label="drift MSE")
plt.plot(cycle_axis, [r["sigma_mse"] for r in cycle_records], marker="o", label="diffusion MSE")
plt.yscale("log")
plt.xticks(cycle_axis)
plt.xlabel("cycle (0 = Stage I)")
plt.ylabel("primary held-out coefficient MSE")
plt.title(f"E3 coefficient recovery — {CONFIG['TRUE_SDE_NAME']}")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(cycle_axis, [r["test_nll"] for r in cycle_records], marker="o")
plt.xticks(cycle_axis)
plt.xlabel("cycle (0 = Stage I)")
plt.ylabel("held-out transition NLL")
plt.title(f"E3 held-out NLL — {CONFIG['TRUE_SDE_NAME']}")
plt.grid(True)
plt.show()

if len(cycle_records) > 2:
    comparisons = [f"W{k-1}->W{k}" for k in range(2, len(cycle_records))]
    means = [cycle_records[k]["angle_mean_deg"] for k in range(2, len(cycle_records))]
    maxima = [cycle_records[k]["angle_max_deg"] for k in range(2, len(cycle_records))]

    x = np.arange(len(comparisons))
    width = 0.35
    plt.figure(figsize=(7, 4))
    plt.bar(x - width/2, means, width, label="mean angle")
    plt.bar(x + width/2, maxima, width, label="max angle")
    plt.xticks(x, comparisons)
    plt.ylabel("principal angle (degrees)")
    plt.title("Successive learned generator subspaces")
    plt.legend()
    plt.grid(True, axis="y")
    plt.show()
